# Training notebook for ResNet18

This notebook implements training for ResNet18 architectures for image classification.



In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Imports

In [3]:
import os
import random
import csv
import time

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from torchvision import models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset


from tqdm import tqdm
import time

from PIL import Image, ImageFilter
import cv2


## Reproducibility

In [4]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [6]:
zip_path = "/content/drive/MyDrive/DLprojekt1/archive (12).zip"
!unzip -q "/content/drive/MyDrive/DLprojekt1/archive (12).zip" -d "/content/"

## Paths

In [7]:
DATA_DIR = "/content"

PROJECT_DIR = "/content/drive/MyDrive/DLprojekt1"

MODEL_DIR = os.path.join(PROJECT_DIR, "saved_models")
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints")
RESULT_DIR = os.path.join(PROJECT_DIR, "results")
PLOT_DIR = os.path.join(PROJECT_DIR, "plots")

PATIENCE = 5

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

RESULT_FILE = os.path.join(RESULT_DIR, "experiment_results_resnet.csv")

if not os.path.exists(RESULT_FILE):

    with open(RESULT_FILE,"w",newline="") as f:

        writer = csv.writer(f)
        writer.writerow([
            "experiment",
            "lr",
            "optimizer",
            "dropout",
            "weight_decay",
            "augmentation",
            "test_accuracy"
        ])


## Data Augmentations

In [8]:
class SobelAugmentation:

    def __init__(self, alpha=0.5):
        self.alpha = alpha

    def __call__(self, img):

        img_np = np.array(img)

        gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)

        sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
        sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)

        sobel = np.sqrt(sobelx**2 + sobely**2)

        sobel = np.uint8(255 * sobel / np.max(sobel))

        sobel_rgb = np.stack([sobel]*3, axis=-1)

        blended = cv2.addWeighted(img_np, 1-self.alpha, sobel_rgb, self.alpha, 0)

        return Image.fromarray(blended)


class GaussianBlur(object):

    def __call__(self,img):

        return img.filter(ImageFilter.GaussianBlur(radius=1.0))


def get_transform(augmentation=None, model_type="resnet", train=True):

    transform_list = []

    if train:
        transform_list.extend([
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
        ])

    if augmentation == "rotation":
        transform_list.append(transforms.RandomRotation(15))

    elif augmentation == "color_jitter":
        transform_list.append(
            transforms.ColorJitter(
                brightness=0.2,
                contrast=0.2,
                saturation=0.2,
                hue=0.1
            )
        )

    elif augmentation == "gaussian_blur":
        transform_list.append(GaussianBlur())

    elif augmentation == "sobel":
        transform_list.append(SobelAugmentation())

    if model_type == "resnet":
        transform_list.extend([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(
                (0.485, 0.456, 0.406),
                (0.229, 0.224, 0.225)
            )
        ])
    else:
        transform_list.extend([
            transforms.ToTensor(),
            transforms.Normalize(
                (0.4789, 0.4723, 0.4305),
                (0.2421, 0.2383, 0.2587)
            )
        ])

    return transforms.Compose(transform_list)


def mixup_data(x, y, alpha=0.2, device=None):

    lam = np.random.beta(alpha, alpha)

    batch_size = x.size(0)

    if device is None:
        device = x.device

    index = torch.randperm(batch_size).to(device)

    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]

    return mixed_x, y_a, y_b, lam


def mixup_loss(criterion, pred, y_a, y_b, lam):

    return lam*criterion(pred, y_a) + (1-lam)*criterion(pred, y_b)


## Data Loading

In [9]:
def load_data(DATA_DIR, augmentation=None, batch_size=512, subset_ratio=1.0, model_type="resnet"):

    train_transform = get_transform(augmentation=augmentation, model_type=model_type, train=True)
    test_transform = get_transform(augmentation=None, model_type=model_type, train=False)

    train_set = torchvision.datasets.ImageFolder(
        os.path.join(DATA_DIR, "train"),
        transform=train_transform
    )

    valid_set = torchvision.datasets.ImageFolder(
        os.path.join(DATA_DIR, "valid"),
        transform=test_transform
    )

    test_set = torchvision.datasets.ImageFolder(
        os.path.join(DATA_DIR, "test"),
        transform=test_transform
    )

    if subset_ratio < 1.0:
        subset_size = int(len(train_set) * subset_ratio)
        indices = torch.randperm(len(train_set))[:subset_size]
        train_set = Subset(train_set, indices)

    train_loader = DataLoader(
        train_set,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    valid_loader = DataLoader(
        valid_set,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_set,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    return train_loader, valid_loader, test_loader

## ResNet18

In [10]:
class ResNet18Model(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.model = models.resnet18(weights="IMAGENET1K_V1")

        in_features = self.model.fc.in_features

        self.model.fc = nn.Sequential(
            nn.BatchNorm1d(in_features),
            nn.Dropout(dropout),
            nn.Linear(in_features, 10)
        )

    def forward(self, x):
        return self.model(x)

## Evaluation

In [11]:
def evaluate(model, loader):

    model.eval()
    device = next(model.parameters()).device

    total_loss = 0.0
    total_correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(images)
            loss = F.cross_entropy(logits, labels)

            total_loss += loss.item() * labels.size(0)
            total_correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)

    return total_correct / total, total_loss / total

In [12]:
class EarlyStopping:

    def __init__(self,patience=5):
        self.patience = patience
        self.best_score = None
        self.counter = 0
        self.stop = False

    def step(self,score):

        if self.best_score is None:
            self.best_score = score

        elif score <= self.best_score:

            self.counter += 1

            if self.counter >= self.patience:
                self.stop = True

        else:

            self.best_score = score
            self.counter = 0

        return self.stop

## Training

In [13]:
from tqdm import tqdm

def train_model(model, train_loader, val_loader, optimizer, scheduler,
                epochs, use_mixup=False, name="model"):

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler = torch.cuda.amp.GradScaler()

    early_stopping = EarlyStopping(patience=5)

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    best_acc = 0
    model.to(device)

    for epoch in range(epochs):

        model.train()

        running_loss = 0.0
        correct = 0
        total = 0

        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=True)

        for images, labels in loop:

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast():

                if use_mixup:
                    images, y_a, y_b, lam = mixup_data(images, labels)

                outputs = model(images)

                if use_mixup:
                    loss = mixup_loss(criterion, outputs, y_a, y_b, lam)
                else:
                    loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()

            preds = outputs.argmax(1)

            if use_mixup:
                correct += (
                    lam * preds.eq(y_a).sum().item()
                    + (1 - lam) * preds.eq(y_b).sum().item()
                )
            else:
                correct += (preds == labels).sum().item()

            total += labels.size(0)

            loop.set_postfix(
                loss=f"{loss.item():.3f}",
                acc=f"{correct/total:.3f}"
            )

        scheduler.step()

        train_loss = running_loss / len(train_loader)
        train_acc = correct / total

        val_acc, val_loss = evaluate(model, val_loader)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(
            f"\n Epoch {epoch+1}/{epochs}\n"
            f"   Train → loss: {train_loss:.4f}, acc: {train_acc:.4f}\n"
            f"   Val   → loss: {val_loss:.4f}, acc: {val_acc:.4f}\n"
        )

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(
                model.state_dict(),
                os.path.join(CHECKPOINT_DIR, f"best_model_{name}.pt")
            )

        if early_stopping.step(val_acc):
            print("Early stopping triggered")
            break

    return history

## Run Experiment

In [14]:
def save_history_csv(history, experiment_name):

    filename = os.path.join(RESULT_DIR, f"{experiment_name}_history.csv")

    epochs = len(history['train_loss'])

    with open(filename, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['epoch', 'train_loss', 'train_acc', 'val_loss', 'val_acc'])

        for i in range(epochs):
            writer.writerow([
                i+1,
                history['train_loss'][i],
                history['train_acc'][i],
                history['val_loss'][i],
                history['val_acc'][i]
            ])

    print(f"History saved in: {filename}")

In [15]:
def run_experiment(
    model_class,
    name,
    lr=0.01,
    optimizer_name="SGD",
    dropout=0.4,
    weight_decay=5e-4,
    augmentation=None,
    epochs=10,
    subset_ratio=1.0,
    use_mixup=False
):

    print(f"\n Running experiment: {name}")

    train_loader, val_loader, test_loader = load_data(
        DATA_DIR,
        augmentation=augmentation,
        batch_size=512,
        subset_ratio=subset_ratio,
        model_type="resnet"
    )

    model = model_class(dropout=dropout).to(device)

    if optimizer_name == "SGD":
        optimizer = optim.SGD(
            model.parameters(),
            lr=lr,
            weight_decay=weight_decay
        )

    elif optimizer_name == "SGD_momentum":
        optimizer = optim.SGD(
            model.parameters(),
            lr=lr,
            momentum=0.9,
            weight_decay=weight_decay
        )

    elif optimizer_name == "Adam":
        optimizer = optim.Adam(
            model.parameters(),
            lr=lr,
            weight_decay=weight_decay
        )

    else:
        raise ValueError("Unknown optimizer")

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=epochs
    )

    history = train_model(
        model,
        train_loader,
        val_loader,
        optimizer,
        scheduler,
        epochs,
        use_mixup=use_mixup,
        name=name
    )

    test_acc, test_loss = evaluate(model, test_loader)

    print(f"\n Test Accuracy: {test_acc:.4f}")
    print(f" Test Loss: {test_loss:.4f}")

    torch.save(
        model.state_dict(),
        os.path.join(MODEL_DIR, f"{name}.pt")
    )

    save_history_csv(history, name)

    return history, test_acc

## Experiments

Experiments with different training, regularization parameters and data augumentation methods

In [ ]:
run_experiment(
    model_class=ResNet18Model,
    name="resnet_0.1",
    lr=0.1,
    optimizer_name="SGD",
    dropout=0.3,
    weight_decay=1e-4,
    augmentation="None",
    epochs=7,
    subset_ratio=1.0,
    use_mixup=False
)


🚀 Running experiment: resnet_0.1


/tmp/ipykernel_2333/2455028956.py:7: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Epoch 1/7:   0%|          | 0/176 [00:00<?, ?it/s]/tmp/ipykernel_2333/2455028956.py:38: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/7: 100%|██████████| 176/176 [03:44<00:00,  1.27s/it, acc=0.776, loss=0.878]



📊 Epoch 1/7
   Train → loss: 1.0383, acc: 0.7762
   Val   → loss: 0.5307, acc: 0.8352



Epoch 2/7: 100%|██████████| 176/176 [03:41<00:00,  1.26s/it, acc=0.851, loss=0.864]



📊 Epoch 2/7
   Train → loss: 0.8736, acc: 0.8509
   Val   → loss: 0.4716, acc: 0.8567



...

Epoch 6/7: 100%|██████████| 176/176 [03:58<00:00,  1.36s/it, acc=0.906, loss=0.800]



📊 Epoch 6/7
   Train → loss: 0.7560, acc: 0.9061
   Val   → loss: 0.3991, acc: 0.8850



Epoch 7/7: 100%|██████████| 176/176 [03:45<00:00,  1.28s/it, acc=0.911, loss=0.732]



📊 Epoch 7/7
   Train → loss: 0.7476, acc: 0.9109
   Val   → loss: 0.3936, acc: 0.8867


✅ Test Accuracy: 0.8853
✅ Test Loss: 0.3970
History saved in: /content/drive/MyDrive/DLprojekt1/results/resnet_0.1_history.csv


({'train_loss': [1.0383212898265233,
   0.8736088766970418,
   0.8252599374814467,
   0.7954296175051819,
   0.7714607492089272,
   0.7560379725288261,
   0.7475687319582159],
  'train_acc': [0.7762,
   0.8509,
   0.8735444444444445,
   0.8880222222222223,
   0.9003222222222222,
   0.9060666666666667,
   0.9109],
  'val_loss': [0.5307152051713732,
   0.47162698658307395,
   0.4480368403010898,
   0.41968546012242636,
   0.40796996059947543,
   0.3991388770421346,
   0.3936455182976193],
  'val_acc': [0.8352,
   0.8567222222222223,
   0.8670111111111111,
   0.8776444444444444,
   0.8803666666666666,
   0.8849666666666667,
   0.8867444444444444]},
 0.8852666666666666)

In [ ]:
run_experiment(
    model_class=ResNet18Model,
    name="resnet_0.01",
    lr=0.01,
    optimizer_name="SGD",
    dropout=0.3,
    weight_decay=1e-4,
    augmentation="None",
    epochs=7,
    subset_ratio=1.0,
    use_mixup=False
)


🚀 Running experiment: resnet_0.01


/tmp/ipykernel_2333/2455028956.py:7: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Epoch 1/7:   0%|          | 0/176 [00:00<?, ?it/s]/tmp/ipykernel_2333/2455028956.py:38: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/7: 100%|██████████| 176/176 [03:42<00:00,  1.27s/it, acc=0.603, loss=1.076]



📊 Epoch 1/7
   Train → loss: 1.4026, acc: 0.6033
   Val   → loss: 0.7535, acc: 0.7574



Epoch 2/7: 100%|██████████| 176/176 [03:43<00:00,  1.27s/it, acc=0.764, loss=0.996]



📊 Epoch 2/7
   Train → loss: 1.0655, acc: 0.7639
   Val   → loss: 0.6264, acc: 0.8025



...

Epoch 6/7: 100%|██████████| 176/176 [03:44<00:00,  1.28s/it, acc=0.822, loss=0.983]



📊 Epoch 6/7
   Train → loss: 0.9444, acc: 0.8218
   Val   → loss: 0.5392, acc: 0.8335



Epoch 7/7: 100%|██████████| 176/176 [03:43<00:00,  1.27s/it, acc=0.824, loss=0.894]



📊 Epoch 7/7
   Train → loss: 0.9417, acc: 0.8237
   Val   → loss: 0.5370, acc: 0.8338


✅ Test Accuracy: 0.8326
✅ Test Loss: 0.5417
History saved in: /content/drive/MyDrive/DLprojekt1/results/resnet_0.01_history.csv


({'train_loss': [1.4025587696920743,
   1.06553249230439,
   1.0016217621212655,
   0.9706818126142025,
   0.9538285860961134,
   0.9443776224824515,
   0.9417117004367438],
  'train_acc': [0.6033333333333334,
   0.7639333333333334,
   0.7957777777777778,
   0.8093333333333333,
   0.8166,
   0.8217777777777778,
   0.8237333333333333],
  'val_loss': [0.7535432924800449,
   0.6263890743573507,
   0.6057696553972033,
   0.5587612998962402,
   0.5488566032091776,
   0.5391834277047052,
   0.5369833847469754],
  'val_acc': [0.7573555555555556,
   0.8024888888888889,
   0.8108888888888889,
   0.8263555555555555,
   0.8302333333333334,
   0.8335222222222223,
   0.8337666666666667]},
 0.8326)

In [ ]:
run_experiment(
    model_class=ResNet18Model,
    name="resnet_0.001",
    lr=0.001,
    optimizer_name="SGD",
    dropout=0.3,
    weight_decay=1e-4,
    augmentation="None",
    epochs=7,
    subset_ratio=1.0,
    use_mixup=False
)


🚀 Running experiment: resnet_0.001


/tmp/ipykernel_2333/2455028956.py:7: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Epoch 1/7:   0%|          | 0/176 [00:00<?, ?it/s]/tmp/ipykernel_2333/2455028956.py:38: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/7: 100%|██████████| 176/176 [03:42<00:00,  1.27s/it, acc=0.258, loss=1.840]



📊 Epoch 1/7
   Train → loss: 2.1262, acc: 0.2583
   Val   → loss: 1.6903, acc: 0.4328



Epoch 2/7: 100%|██████████| 176/176 [03:58<00:00,  1.35s/it, acc=0.463, loss=1.564]



📊 Epoch 2/7
   Train → loss: 1.7031, acc: 0.4630
   Val   → loss: 1.3401, acc: 0.5687



...

Epoch 6/7: 100%|██████████| 176/176 [03:53<00:00,  1.33s/it, acc=0.640, loss=1.300]



📊 Epoch 6/7
   Train → loss: 1.3292, acc: 0.6403
   Val   → loss: 1.0147, acc: 0.6773



Epoch 7/7: 100%|██████████| 176/176 [03:50<00:00,  1.31s/it, acc=0.644, loss=1.315]



📊 Epoch 7/7
   Train → loss: 1.3210, acc: 0.6445
   Val   → loss: 1.0096, acc: 0.6787


✅ Test Accuracy: 0.6784
✅ Test Loss: 1.0140
History saved in: /content/drive/MyDrive/DLprojekt1/results/resnet_0.001_history.csv


({'train_loss': [2.1261615949598225,
   1.70312068679116,
   1.506010228259997,
   1.4063765237277204,
   1.353463321246884,
   1.3292126892642542,
   1.321042762561278],
  'train_acc': [0.25826666666666664,
   0.4629888888888889,
   0.5572444444444444,
   0.6039777777777777,
   0.6296,
   0.6402888888888889,
   0.6444666666666666],
  'val_loss': [1.69030139986674,
   1.3400942432403564,
   1.1671716393788656,
   1.080842447566986,
   1.0363145238982308,
   1.0146929949866401,
   1.0096028802977668],
  'val_acc': [0.4328111111111111,
   0.5687111111111111,
   0.6275333333333334,
   0.6553666666666667,
   0.6704888888888889,
   0.6773222222222223,
   0.6786777777777778]},
 0.6784444444444444)

In [ ]:
### to juz mamy -> SGD
#run_experiment(
#    model_class=ResNet18Model,
#    name="resnet_0.1",
#    lr=0.1,
#    optimizer_name="SGD",
#    dropout=0.3,
#    weight_decay=1e-4,
#    augmentation="None",
#    epochs=7,
#    subset_ratio=1.0,
#    use_mixup=False
#)

In [ ]:
run_experiment(
    model_class=ResNet18Model,
    name="resnet_sgd_momentum",
    lr=0.1,
    optimizer_name="SGD_momentum",
    dropout=0.3,
    weight_decay=1e-4,
    augmentation="None",
    epochs=7,
    subset_ratio=1.0,
    use_mixup=False
)


 Running experiment: resnet_sgd_momentum
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 132MB/s]
/tmp/ipykernel_1216/434444978.py:7: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Epoch 1/7:   0%|          | 0/176 [00:00<?, ?it/s]/tmp/ipykernel_1216/434444978.py:38: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/7: 100%|██████████| 176/176 [03:51<00:00,  1.31s/it, acc=0.729, loss=0.981]



 Epoch 1/7
   Train → loss: 1.1588, acc: 0.7289
   Val   → loss: 0.5839, acc: 0.8130



Epoch 2/7: 100%|██████████| 176/176 [03:31<00:00,  1.20s/it, acc=0.831, loss=0.890]



 Epoch 2/7
   Train → loss: 0.9189, acc: 0.8313
   Val   → loss: 0.5269, acc: 0.8345



...

Epoch 6/7: 100%|██████████| 176/176 [03:19<00:00,  1.13s/it, acc=0.942, loss=0.650]



 Epoch 6/7
   Train → loss: 0.6699, acc: 0.9416
   Val   → loss: 0.3752, acc: 0.8902



Epoch 7/7: 100%|██████████| 176/176 [03:23<00:00,  1.15s/it, acc=0.954, loss=0.672]



 Epoch 7/7
   Train → loss: 0.6449, acc: 0.9536
   Val   → loss: 0.3648, acc: 0.8955


 Test Accuracy: 0.8940
 Test Loss: 0.3678
History saved in: /content/drive/MyDrive/DLprojekt1/results/resnet_sgd_momentum_history.csv


({'train_loss': [1.1587706980380146,
   0.9188609336587515,
   0.8350586447526108,
   0.7706903046505018,
   0.7153064401989634,
   0.6699267879805781,
   0.6448814858767119],
  'train_acc': [0.7289444444444444,
   0.8313111111111111,
   0.8664444444444445,
   0.8949222222222222,
   0.9202111111111111,
   0.9415555555555556,
   0.9536222222222223],
  'val_loss': [0.5839085858133104,
   0.5268842214478386,
   0.4987046470748054,
   0.45239076843261716,
   0.39974401921696134,
   0.3752199385325114,
   0.36480330233044095],
  'val_acc': [0.813,
   0.8345444444444444,
   0.8455111111111111,
   0.8653777777777778,
   0.8826666666666667,
   0.8901888888888889,
   0.8955222222222222]},
 0.894)

In [ ]:
run_experiment(
    model_class=ResNet18Model,
    name="resnet_adam",
    lr=0.001,
    optimizer_name="Adam",
    dropout=0.3,
    weight_decay=1e-4,
    augmentation="None",
    epochs=7,
    subset_ratio=1.0,
    use_mixup=False
)


 Running experiment: resnet_adam


Exception ignored in: <function _ConnectionBase.__del__ at 0x79822102d9e0>
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 133, in __del__
    self._close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 377, in _close
    _close(self._handle)
OSError: [Errno 9] Bad file descriptor
/tmp/ipykernel_1216/434444978.py:7: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Epoch 1/7:   0%|          | 0/176 [00:00<?, ?it/s]/tmp/ipykernel_1216/434444978.py:38: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/7: 100%|██████████| 176/176 [03:21<00:00,  1.15s/it, acc=0.757, loss=1.001]



 Epoch 1/7
   Train → loss: 1.0667, acc: 0.7566
   Val   → loss: 0.6627, acc: 0.7841



Epoch 2/7: 100%|██████████| 176/176 [03:25<00:00,  1.17s/it, acc=0.832, loss=0.889]



 Epoch 2/7
   Train → loss: 0.8979, acc: 0.8325
   Val   → loss: 0.6313, acc: 0.7960



...

Epoch 6/7: 100%|██████████| 176/176 [03:24<00:00,  1.16s/it, acc=0.944, loss=0.608]



 Epoch 6/7
   Train → loss: 0.6472, acc: 0.9436
   Val   → loss: 0.3948, acc: 0.8844



Epoch 7/7: 100%|██████████| 176/176 [03:36<00:00,  1.23s/it, acc=0.964, loss=0.577]



 Epoch 7/7
   Train → loss: 0.6056, acc: 0.9641
   Val   → loss: 0.3734, acc: 0.8928


 Test Accuracy: 0.8919
 Test Loss: 0.3757
History saved in: /content/drive/MyDrive/DLprojekt1/results/resnet_adam_history.csv


({'train_loss': [1.0667398754845967,
   0.8979296552186663,
   0.8309555659917268,
   0.7694454707882621,
   0.7075731334361163,
   0.6472287787632509,
   0.6055875413797118],
  'train_acc': [0.7566222222222222,
   0.8324888888888888,
   0.8598333333333333,
   0.8880666666666667,
   0.9160777777777778,
   0.9436444444444444,
   0.9640777777777778],
  'val_loss': [0.6627337370554606,
   0.6313289744695028,
   0.6423196943283082,
   0.4971006133821276,
   0.4351878249062432,
   0.3947557494375441,
   0.3733973846859402],
  'val_acc': [0.7840555555555555,
   0.796,
   0.7945333333333333,
   0.8444666666666667,
   0.8671444444444445,
   0.8844333333333333,
   0.8927777777777778]},
 0.8919222222222222)

In [ ]:
run_experiment(
    model_class=ResNet18Model,
    name="resnet_dropout_0.1",
    lr=0.1,
    optimizer_name="SGD_momentum",
    dropout=0.1,
    weight_decay=1e-4,
    augmentation="None",
    epochs=7,
    subset_ratio=1.0,
    use_mixup=False
)


 Running experiment: resnet_dropout_0.1


/tmp/ipykernel_1216/434444978.py:7: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Epoch 1/7:   0%|          | 0/176 [00:00<?, ?it/s]/tmp/ipykernel_1216/434444978.py:38: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/7: 100%|██████████| 176/176 [03:44<00:00,  1.28s/it, acc=0.743, loss=0.930]



 Epoch 1/7
   Train → loss: 1.1211, acc: 0.7426
   Val   → loss: 0.6424, acc: 0.7980



Epoch 2/7: 100%|██████████| 176/176 [03:23<00:00,  1.16s/it, acc=0.842, loss=0.939]



 Epoch 2/7
   Train → loss: 0.8920, acc: 0.8421
   Val   → loss: 0.5625, acc: 0.8196



...

Epoch 6/7: 100%|██████████| 176/176 [03:27<00:00,  1.18s/it, acc=0.949, loss=0.656]



 Epoch 6/7
   Train → loss: 0.6529, acc: 0.9487
   Val   → loss: 0.3788, acc: 0.8903



Epoch 7/7: 100%|██████████| 176/176 [03:24<00:00,  1.16s/it, acc=0.961, loss=0.643]



 Epoch 7/7
   Train → loss: 0.6284, acc: 0.9607
   Val   → loss: 0.3621, acc: 0.8969


 Test Accuracy: 0.8941
 Test Loss: 0.3666
History saved in: /content/drive/MyDrive/DLprojekt1/results/resnet_dropout_0.1_history.csv


({'train_loss': [1.1210685473951427,
   0.891998733647845,
   0.8084497001360763,
   0.7466845349832014,
   0.6944003027271141,
   0.6528515934266828,
   0.6283934062177484],
  'train_acc': [0.7426444444444444,
   0.8421333333333333,
   0.8775,
   0.9053333333333333,
   0.9286333333333333,
   0.9486666666666667,
   0.9607444444444444],
  'val_loss': [0.6423713272306654,
   0.5625342656347486,
   0.5217582896550497,
   0.43560959027608237,
   0.393195386229621,
   0.37876224046283297,
   0.3620622524526384],
  'val_acc': [0.7980444444444444,
   0.8195555555555556,
   0.8390555555555556,
   0.8654666666666667,
   0.8832666666666666,
   0.8903444444444445,
   0.8968777777777778]},
 0.8941111111111111)

In [ ]:
### to juz mamy
#run_experiment(
#    model_class=ResNet18Model,
#    name="resnet_dropout_0.3",
#    lr=0.1,
#    optimizer_name="SGD_momentum",
#    dropout=0.3,
#    weight_decay=1e-4,
#    augmentation="None",
#    epochs=7,
#    subset_ratio=1.0,
#    use_mixup=False
#)

In [ ]:
run_experiment(
    model_class=ResNet18Model,
    name="resnet_dropout_0.5",
    lr=0.1,
    optimizer_name="SGD_momentum",
    dropout=0.5,
    weight_decay=1e-4,
    augmentation="None",
    epochs=7,
    subset_ratio=1.0,
    use_mixup=False
)


 Running experiment: resnet_dropout_0.5
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 136MB/s]
/tmp/ipykernel_610/434444978.py:7: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Epoch 1/7:   0%|          | 0/176 [00:00<?, ?it/s]/tmp/ipykernel_610/434444978.py:38: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/7: 100%|██████████| 176/176 [03:57<00:00,  1.35s/it, acc=0.701, loss=1.015]



 Epoch 1/7
   Train → loss: 1.2471, acc: 0.7008
   Val   → loss: 0.6631, acc: 0.7789



Epoch 2/7: 100%|██████████| 176/176 [03:37<00:00,  1.24s/it, acc=0.822, loss=0.927]



 Epoch 2/7
   Train → loss: 0.9509, acc: 0.8217
   Val   → loss: 0.6382, acc: 0.7922



...

Epoch 6/7: 100%|██████████| 176/176 [03:25<00:00,  1.17s/it, acc=0.933, loss=0.678]



 Epoch 6/7
   Train → loss: 0.6890, acc: 0.9334
   Val   → loss: 0.3885, acc: 0.8842



Epoch 7/7: 100%|██████████| 176/176 [03:24<00:00,  1.16s/it, acc=0.946, loss=0.672]



 Epoch 7/7
   Train → loss: 0.6630, acc: 0.9460
   Val   → loss: 0.3745, acc: 0.8922


 Test Accuracy: 0.8906
 Test Loss: 0.3780
History saved in: /content/drive/MyDrive/DLprojekt1/results/resnet_dropout_0.5_history.csv


({'train_loss': [1.2470569800246845,
   0.9508533738553524,
   0.8627649623561989,
   0.7944173690947619,
   0.7362988879057494,
   0.6889947581697594,
   0.6629894998940554],
  'train_acc': [0.7008444444444445,
   0.8216666666666667,
   0.8583111111111111,
   0.8870333333333333,
   0.912,
   0.9334444444444444,
   0.9460444444444445],
  'val_loss': [0.6631471910688612,
   0.6381951753987206,
   0.4834949284447564,
   0.45477578880521985,
   0.41294725466834176,
   0.38853342634836835,
   0.37454725878503586],
  'val_acc': [0.7788555555555555,
   0.7922333333333333,
   0.8482777777777778,
   0.8610777777777778,
   0.8760777777777777,
   0.8842333333333333,
   0.8921666666666667]},
 0.8906)

In [ ]:
run_experiment(
    model_class=ResNet18Model,
    name="resnet_weight_decay1e02",
    lr=0.1,
    optimizer_name="SGD_momentum",
    dropout=0.3,
    weight_decay=1e-2,
    augmentation="None",
    epochs=7,
    subset_ratio=1.0,
    use_mixup=False
)


 Running experiment: resnet_weight_decay1e02


/tmp/ipykernel_610/434444978.py:7: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Epoch 1/7:   0%|          | 0/176 [00:00<?, ?it/s]/tmp/ipykernel_610/434444978.py:38: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/7: 100%|██████████| 176/176 [03:29<00:00,  1.19s/it, acc=0.660, loss=1.333]



 Epoch 1/7
   Train → loss: 1.2768, acc: 0.6603
   Val   → loss: 1.9440, acc: 0.3199



Epoch 2/7: 100%|██████████| 176/176 [03:27<00:00,  1.18s/it, acc=0.609, loss=1.397]



 Epoch 2/7
   Train → loss: 1.3707, acc: 0.6089
   Val   → loss: 1.8861, acc: 0.3601



...

Epoch 6/7: 100%|██████████| 176/176 [03:25<00:00,  1.17s/it, acc=0.724, loss=1.105]



 Epoch 6/7
   Train → loss: 1.1261, acc: 0.7241
   Val   → loss: 1.0123, acc: 0.6598



Epoch 7/7: 100%|██████████| 176/176 [03:23<00:00,  1.16s/it, acc=0.780, loss=1.004]



 Epoch 7/7
   Train → loss: 1.0064, acc: 0.7803
   Val   → loss: 0.7076, acc: 0.7769


 Test Accuracy: 0.7738
 Test Loss: 0.7167
History saved in: /content/drive/MyDrive/DLprojekt1/results/resnet_weight_decay1e02_history.csv


({'train_loss': [1.2767920527945866,
   1.3706633062525229,
   1.3637985329736362,
   1.3097104491157965,
   1.2311906848441472,
   1.1261305565183812,
   1.006376775489612],
  'train_acc': [0.6602777777777777,
   0.6088666666666667,
   0.6124666666666667,
   0.6364888888888889,
   0.6755111111111111,
   0.7241,
   0.7802555555555556],
  'val_loss': [1.94399864209493,
   1.8861341108957927,
   1.5980591385099623,
   1.6153588844299316,
   1.1618197484546238,
   1.0122536261452568,
   0.7075801855934991],
  'val_acc': [0.31987777777777776,
   0.36006666666666665,
   0.4465,
   0.4524888888888889,
   0.6047777777777777,
   0.6598222222222222,
   0.7768555555555555]},
 0.7738)

In [ ]:
run_experiment(
    model_class=ResNet18Model,
    name="resnet_weight_decay1e03",
    lr=0.1,
    optimizer_name="SGD_momentum",
    dropout=0.3,
    weight_decay=1e-3,
    augmentation="None",
    epochs=7,
    subset_ratio=1.0,
    use_mixup=False
)


 Running experiment: resnet_weight_decay1e03


/tmp/ipykernel_610/434444978.py:7: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Epoch 1/7:   0%|          | 0/176 [00:00<?, ?it/s]/tmp/ipykernel_610/434444978.py:38: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/7: 100%|██████████| 176/176 [03:25<00:00,  1.17s/it, acc=0.726, loss=1.030]



 Epoch 1/7
   Train → loss: 1.1679, acc: 0.7256
   Val   → loss: 0.7256, acc: 0.7604



Epoch 2/7: 100%|██████████| 176/176 [03:23<00:00,  1.16s/it, acc=0.823, loss=0.951]



 Epoch 2/7
   Train → loss: 0.9273, acc: 0.8228
   Val   → loss: 0.6117, acc: 0.8100



...

Epoch 6/7: 100%|██████████| 176/176 [03:26<00:00,  1.17s/it, acc=0.932, loss=0.703]



 Epoch 6/7
   Train → loss: 0.6831, acc: 0.9324
   Val   → loss: 0.4118, acc: 0.8782



Epoch 7/7: 100%|██████████| 176/176 [03:25<00:00,  1.17s/it, acc=0.954, loss=0.627]



 Epoch 7/7
   Train → loss: 0.6381, acc: 0.9540
   Val   → loss: 0.3687, acc: 0.8935


 Test Accuracy: 0.8930
 Test Loss: 0.3726
History saved in: /content/drive/MyDrive/DLprojekt1/results/resnet_weight_decay1e03_history.csv


({'train_loss': [1.1679319204254583,
   0.927301848815246,
   0.8655817393552173,
   0.8137805885211988,
   0.7501440532505512,
   0.6831227486783807,
   0.6380838826298714],
  'train_acc': [0.7255666666666667,
   0.8227666666666666,
   0.8479,
   0.8713222222222222,
   0.8998444444444444,
   0.9323666666666667,
   0.9540111111111111],
  'val_loss': [0.7256426262749566,
   0.6116856418715583,
   0.6271326595518324,
   0.5701571678108639,
   0.4529479030079312,
   0.4118050807952881,
   0.36869652626249527],
  'val_acc': [0.7603888888888889,
   0.8099888888888889,
   0.7969777777777778,
   0.8206222222222223,
   0.8600666666666666,
   0.8781777777777777,
   0.8935333333333333]},
 0.8929777777777778)

In [ ]:
# to mamy
#run_experiment(
#    model_class=ResNet18Model,
#    name="resnet_weight_decay1e04",
#    lr=0.1,
#    optimizer_name="SGD_momentum",
#    dropout=0.3,
#    weight_decay=1e-4,
#    augmentation="None",
#    epochs=7,
#    subset_ratio=1.0,
#    use_mixup=False
#)

DATA AUGUMENTATION EXPERIMENTS

In [ ]:
run_experiment(
    model_class=ResNet18Model,
    name="resnet_augmentation_color_jitter",
    lr=0.1,
    optimizer_name="SGD_momentum",
    dropout=0.3,
    weight_decay=1e-4,
    augmentation="color_jitter",
    epochs=12
)


 Running experiment: resnet_augmentation_color_jitter


/tmp/ipykernel_610/434444978.py:7: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Epoch 1/12:   0%|          | 0/176 [00:00<?, ?it/s]/tmp/ipykernel_610/434444978.py:38: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/12: 100%|██████████| 176/176 [04:19<00:00,  1.47s/it, acc=0.709, loss=0.980]



 Epoch 1/12
   Train → loss: 1.2068, acc: 0.7086
   Val   → loss: 0.6525, acc: 0.7892



Epoch 2/12: 100%|██████████| 176/176 [04:03<00:00,  1.38s/it, acc=0.811, loss=0.935]



 Epoch 2/12
   Train → loss: 0.9700, acc: 0.8111
   Val   → loss: 0.6381, acc: 0.7929



...

Epoch 11/12: 100%|██████████| 176/176 [04:10<00:00,  1.42s/it, acc=0.967, loss=0.626]



 Epoch 11/12
   Train → loss: 0.6133, acc: 0.9667
   Val   → loss: 0.3713, acc: 0.8928



Epoch 12/12: 100%|██████████| 176/176 [04:01<00:00,  1.37s/it, acc=0.970, loss=0.603]



 Epoch 12/12
   Train → loss: 0.6065, acc: 0.9700
   Val   → loss: 0.3684, acc: 0.8943


 Test Accuracy: 0.8945
 Test Loss: 0.3706
History saved in: /content/drive/MyDrive/DLprojekt1/results/resnet_augmentation_color_jitter_history.csv


({'train_loss': [1.2068159783428365,
   0.9700464565645565,
   0.8912828930399634,
   0.8370487229390577,
   0.7959968901493333,
   0.7576600225134329,
   0.7169879038225521,
   0.6826037920334123,
   0.6508717574179173,
   0.6273640038614924,
   0.6132910258390687,
   0.6064565554261208],
  'train_acc': [0.7086222222222223,
   0.8111,
   0.8455444444444444,
   0.8686444444444444,
   0.8850222222222223,
   0.901,
   0.9192222222222223,
   0.9342666666666667,
   0.9490777777777778,
   0.9600444444444445,
   0.9667444444444444,
   0.9699888888888889],
  'val_loss': [0.652457071463267,
   0.6381309360398186,
   0.6034663019074334,
   0.4868203301959568,
   0.4781583327505324,
   0.4580468045764499,
   0.43722719830407036,
   0.4080719144450294,
   0.38700280582639907,
   0.37513041814168296,
   0.37127982032034135,
   0.36835462933646307],
  'val_acc': [0.7891666666666667,
   0.7929111111111111,
   0.8125555555555556,
   0.8506777777777778,
   0.8499222222222222,
   0.8627666666666667,
  

In [17]:
run_experiment(
    model_class=ResNet18Model,
    name="resnet_augmentation_blur",
    lr=0.1,
    optimizer_name="SGD_momentum",
    dropout=0.3,
    weight_decay=1e-4,
    augmentation="gaussian_blur",
    epochs=7
)


 Running experiment: resnet_augmentation_blur
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 190MB/s]
/tmp/ipykernel_5444/434444978.py:7: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Epoch 1/7:   0%|          | 0/176 [00:00<?, ?it/s]/tmp/ipykernel_5444/434444978.py:38: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/7: 100%|██████████| 176/176 [03:47<00:00,  1.29s/it, acc=0.597, loss=1.174]



 Epoch 1/7
   Train → loss: 1.4446, acc: 0.5968
   Val   → loss: 1.3618, acc: 0.5314



Epoch 2/7: 100%|██████████| 176/176 [03:53<00:00,  1.32s/it, acc=0.755, loss=1.031]



 Epoch 2/7
   Train → loss: 1.0864, acc: 0.7550
   Val   → loss: 1.8654, acc: 0.4073



...

Epoch 5/7: 100%|██████████| 176/176 [03:29<00:00,  1.19s/it, acc=0.867, loss=0.853]



 Epoch 5/7
   Train → loss: 0.8308, acc: 0.8666
   Val   → loss: 2.1328, acc: 0.2580



Epoch 6/7: 100%|██████████| 176/176 [03:29<00:00,  1.19s/it, acc=0.894, loss=0.753]



 Epoch 6/7
   Train → loss: 0.7727, acc: 0.8939
   Val   → loss: 2.2751, acc: 0.2810

Early stopping triggered

 Test Accuracy: 0.2749
 Test Loss: 2.2807
History saved in: /content/drive/MyDrive/DLprojekt1/results/resnet_augmentation_blur_history.csv


({'train_loss': [1.4446266876025633,
   1.0864422429691662,
   0.9772048548541286,
   0.8969180878590454,
   0.8307785164903511,
   0.7727218964560465],
  'train_acc': [0.5967555555555556,
   0.7550111111111111,
   0.8032,
   0.8370555555555556,
   0.8665666666666667,
   0.8939],
  'val_loss': [1.3618239923583138,
   1.865444048055013,
   2.289242756568061,
   1.9522191182878281,
   2.1327658061557346,
   2.275100706524319],
  'val_acc': [0.5314333333333333,
   0.4073,
   0.26113333333333333,
   0.3086,
   0.25795555555555555,
   0.2810111111111111]},
 0.27486666666666665)

In [18]:
run_experiment(
    model_class=ResNet18Model,
    name="resnet_augmentation_sobel",
    lr=0.1,
    optimizer_name="SGD_momentum",
    dropout=0.3,
    weight_decay=1e-4,
    augmentation="sobel",
    epochs=7
)


 Running experiment: resnet_augmentation_sobel


/tmp/ipykernel_1268/434444978.py:7: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Epoch 1/7:   0%|          | 0/176 [00:00<?, ?it/s]/tmp/ipykernel_1268/434444978.py:38: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/7: 100%|██████████| 176/176 [03:55<00:00,  1.34s/it, acc=0.680, loss=1.123]



 Epoch 1/7
   Train → loss: 1.2627, acc: 0.6799
   Val   → loss: 0.8534, acc: 0.7208



Epoch 2/7: 100%|██████████| 176/176 [04:08<00:00,  1.41s/it, acc=0.795, loss=1.023]



 Epoch 2/7
   Train → loss: 0.9963, acc: 0.7949
   Val   → loss: 0.8278, acc: 0.7358



...

Epoch 6/7: 100%|██████████| 176/176 [03:55<00:00,  1.34s/it, acc=0.919, loss=0.725]



 Epoch 6/7
   Train → loss: 0.7198, acc: 0.9188
   Val   → loss: 0.6455, acc: 0.7961



Epoch 7/7: 100%|██████████| 176/176 [03:54<00:00,  1.33s/it, acc=0.932, loss=0.717]



 Epoch 7/7
   Train → loss: 0.6924, acc: 0.9317
   Val   → loss: 0.6133, acc: 0.8079


 Test Accuracy: 0.8085
 Test Loss: 0.6134
History saved in: /content/drive/MyDrive/DLprojekt1/results/resnet_augmentation_sobel_history.csv


({'train_loss': [1.2627004994587465,
   0.9963176331736825,
   0.9021977081217549,
   0.8298548002811995,
   0.7717151980508458,
   0.7197545631365343,
   0.692377357320352],
  'train_acc': [0.6799,
   0.7949333333333334,
   0.8362888888888889,
   0.8679888888888889,
   0.8937333333333334,
   0.9187777777777778,
   0.9316555555555556],
  'val_loss': [0.85343020573722,
   0.8278066454357571,
   0.7922849144617716,
   0.7462765017827352,
   0.6326386288113064,
   0.6455392745653789,
   0.6132591130044726],
  'val_acc': [0.7207777777777777,
   0.7357666666666667,
   0.7489444444444444,
   0.7583555555555556,
   0.7985777777777778,
   0.7960555555555555,
   0.8078888888888889]},
 0.8085111111111111)

In [17]:
run_experiment(
    model_class=ResNet18Model,
    name="resnet_augmentation_mixup",
    lr=0.1,
    optimizer_name="SGD_momentum",
    dropout=0.3,
    weight_decay=1e-4,
    use_mixup=True,
    epochs=7
)


 Running experiment: resnet_augmentation_mixup


/tmp/ipykernel_1268/434444978.py:7: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Epoch 1/7:   0%|          | 0/176 [00:00<?, ?it/s]/tmp/ipykernel_1268/434444978.py:38: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/7: 100%|██████████| 176/176 [03:28<00:00,  1.19s/it, acc=0.630, loss=1.019]



 Epoch 1/7
   Train → loss: 1.4531, acc: 0.6300
   Val   → loss: 0.7267, acc: 0.7603



Epoch 2/7: 100%|██████████| 176/176 [03:26<00:00,  1.18s/it, acc=0.733, loss=1.904]



 Epoch 2/7
   Train → loss: 1.2339, acc: 0.7333
   Val   → loss: 0.5893, acc: 0.8184



...

Epoch 6/7: 100%|██████████| 176/176 [03:31<00:00,  1.20s/it, acc=0.820, loss=1.338]



 Epoch 6/7
   Train → loss: 0.9965, acc: 0.8202
   Val   → loss: 0.4711, acc: 0.8860



Epoch 7/7: 100%|██████████| 176/176 [03:34<00:00,  1.22s/it, acc=0.826, loss=0.685]



 Epoch 7/7
   Train → loss: 0.9838, acc: 0.8262
   Val   → loss: 0.4693, acc: 0.8912


 Test Accuracy: 0.8892
 Test Loss: 0.4729
History saved in: /content/drive/MyDrive/DLprojekt1/results/resnet_augmentation_mixup_history.csv


({'train_loss': [1.4530786628072911,
   1.2339091754772447,
   1.1575532907789403,
   1.0978349037468433,
   1.0425534255125306,
   0.9964950067753141,
   0.9837609068913893],
  'train_acc': [0.6300227806439559,
   0.7333252571751621,
   0.7645164358595604,
   0.7817824409339718,
   0.8009073398632727,
   0.8201711843796239,
   0.8261692611351115],
  'val_loss': [0.7267066721757253,
   0.5893213165283203,
   0.6463947531488207,
   0.5446992472118801,
   0.5079705815739102,
   0.47109238289727107,
   0.46930201182895237],
  'val_acc': [0.7603111111111112,
   0.8183666666666667,
   0.8180666666666667,
   0.8654444444444445,
   0.8793333333333333,
   0.8860222222222223,
   0.8912]},
 0.8892111111111111)

In [16]:
run_experiment(
    model_class=ResNet18Model,
    name="resnet_augmentation_rotation",
    lr=0.1,
    optimizer_name="SGD_momentum",
    dropout=0.3,
    weight_decay=1e-4,
    augmentation="rotation",
    epochs=7
)


 Running experiment: resnet_augmentation_rotation
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 191MB/s]
/tmp/ipykernel_1268/434444978.py:7: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Epoch 1/7:   0%|          | 0/176 [00:00<?, ?it/s]/tmp/ipykernel_1268/434444978.py:38: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/7: 100%|██████████| 176/176 [03:55<00:00,  1.34s/it, acc=0.673, loss=1.078]



 Epoch 1/7
   Train → loss: 1.2781, acc: 0.6725
   Val   → loss: 0.6922, acc: 0.7669



Epoch 2/7: 100%|██████████| 176/176 [03:37<00:00,  1.24s/it, acc=0.788, loss=0.969]



 Epoch 2/7
   Train → loss: 1.0153, acc: 0.7883
   Val   → loss: 0.5924, acc: 0.8064



...

Epoch 6/7: 100%|██████████| 176/176 [03:33<00:00,  1.21s/it, acc=0.896, loss=0.756]



 Epoch 6/7
   Train → loss: 0.7620, acc: 0.8960
   Val   → loss: 0.3906, acc: 0.8818



Epoch 7/7: 100%|██████████| 176/176 [03:31<00:00,  1.20s/it, acc=0.907, loss=0.742]



 Epoch 7/7
   Train → loss: 0.7372, acc: 0.9072
   Val   → loss: 0.3812, acc: 0.8878


 Test Accuracy: 0.8848
 Test Loss: 0.3866
History saved in: /content/drive/MyDrive/DLprojekt1/results/resnet_augmentation_rotation_history.csv


({'train_loss': [1.2780696255239574,
   1.0153499228710479,
   0.9273505390367724,
   0.8644963225180452,
   0.8072245662862604,
   0.7619609213010832,
   0.7372197678143327],
  'train_acc': [0.6725444444444444,
   0.7883333333333333,
   0.8237555555555556,
   0.8514,
   0.8758111111111111,
   0.8959666666666667,
   0.9072],
  'val_loss': [0.6922001065995959,
   0.5924473193274604,
   0.5155404783672757,
   0.47874265265994603,
   0.42633398631413777,
   0.3905970234553019,
   0.3811731361071269],
  'val_acc': [0.7669222222222222,
   0.8064333333333333,
   0.8389,
   0.8501,
   0.873,
   0.8818333333333334,
   0.8877555555555555]},
 0.8848444444444444)